# M0 run 2 — MuseTalk, with the three run-1 causes fixed

Read `docs/M0_SPIKE.md` first. Run 1 failed for three reasons and this notebook fixes
each one explicitly:

1. **The inference command was incomplete** (my bug). v1.5 needs `--version v15`,
   `--unet_model_path`, `--unet_config`, and `--ffmpeg_path`.
2. **The weight download exits 0 having downloaded almost nothing.** MuseTalk's script
   points `HF_ENDPOINT` at a mirror, has no `set -e`, and validates nothing. Cell 4 here
   verifies every checkpoint's size and **refuses to continue** if any is missing.
3. **Python 3.12 versus a 3.10-pinned stack.** `mmcv==2.0.1` has no 3.12 wheel, which is
   why run 1's `pip install` finished in 13 seconds without installing OpenMMLab.

**Runtime → Change runtime type → T4 GPU** before you start.

Cell 2 decides Route A (conda, Python 3.10) or Route B (stay on 3.12) for you. Route A
restarts the runtime once — **that is expected, not a crash.**


## 1. What hardware and Python did we get


In [ ]:
import json, os, subprocess, sys, threading, time
from pathlib import Path

LOG, NOTES = {}, []


def note(msg):
    stamp = time.strftime('%H:%M')
    NOTES.append(f'{stamp}  {msg}')
    print(f'noted: {stamp}  {msg}')


def sh(cmd, check=False, quiet=False):
    if not quiet:
        print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout and not quiet:
        print(r.stdout[-3000:])
    if r.returncode != 0 and not quiet:
        print('STDERR:', (r.stderr or '')[-3000:], file=sys.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'failed: {cmd}')
    return r


gpu = sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader')
if gpu.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
name, vram, driver = [f.strip() for f in gpu.stdout.strip().split(',')]

LOG['gpu'] = {'name': name, 'vram_total': vram, 'driver': driver}
LOG['python'] = sys.version.split()[0]
LOG['ffmpeg'] = sh('which ffmpeg', quiet=True).stdout.strip() or None
print(json.dumps(LOG, indent=2))

if LOG['ffmpeg'] is None:
    note('ffmpeg missing -- installing; MuseTalk requires it at inference time')
    sh('apt-get -qq install -y ffmpeg')
    LOG['ffmpeg'] = sh('which ffmpeg', quiet=True).stdout.strip()
print('ffmpeg:', LOG['ffmpeg'])


## 2. Route A or Route B

MuseTalk pins Python 3.10 + torch 2.0.1 + mmcv 2.0.1 + mmdet 3.1.0 + mmpose 1.1.0.
`mmcv==2.0.1` has no wheel for Python 3.12, so on a 3.12 runtime it cannot install --
which is exactly what happened in run 1, silently.


In [ ]:
major, minor = sys.version_info[:2]
NEEDS_CONDA = (major, minor) != (3, 10)

print(f'runtime Python: {major}.{minor}')
if NEEDS_CONDA:
    print('\n  -> ROUTE A recommended: install conda and build a Python 3.10 env.')
    print('     Run cell 3A. The runtime WILL restart -- that is expected.')
    print('     After it restarts, re-run cells 1 and 2, then go to cell 3A-part-2.')
    print('\n  -> ROUTE B (faster, may fail): skip to cell 3B and try newer mmcv on 3.12.')
else:
    print('\n  -> Python 3.10 already. Skip to cell 3B; the pinned stack should install.')
LOG['route'] = 'A (conda needed)' if NEEDS_CONDA else 'B (native 3.10)'


## 3A. Route A — conda, part 1 (restarts the runtime)

Run this **only** if cell 2 said Route A. The restart is normal. When it comes back,
re-run cells 1 and 2, then continue at 3A part 2 — **do not run 3A part 1 twice.**


In [ ]:
# ROUTE A PART 1 -- causes a runtime restart.
!pip install -q condacolab
import condacolab
condacolab.install()   # runtime restarts here


### 3A part 2 — create the pinned environment

Run after the restart. ~8 minutes: conda solves the environment, then the OpenMMLab stack
installs through `openmim`, which is the part that cannot work on 3.12.


In [ ]:
WORK = Path('/content/m0')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

if not (WORK / 'MuseTalk').exists():
    t = time.perf_counter()
    sh('git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git', check=True)
    LOG['clone_s'] = round(time.perf_counter() - t, 1)
os.chdir(WORK / 'MuseTalk')
LOG['commit'] = sh('git rev-parse --short HEAD', quiet=True).stdout.strip()
print('commit', LOG['commit'])

ENV = 'musetalk'
t = time.perf_counter()
sh(f'conda create -y -n {ENV} python=3.10 2>&1 | tail -5')

# Everything below runs inside the 3.10 env. `conda run` keeps that explicit rather than
# relying on shell activation, which does not persist between notebook cells.
R = f'conda run -n {ENV} --no-capture-output'
sh(f'{R} pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 '
   f'--index-url https://download.pytorch.org/whl/cu118 2>&1 | tail -5')
sh(f'{R} pip install -q -r requirements.txt 2>&1 | tail -10')
sh(f'{R} pip install -q --no-cache-dir -U openmim 2>&1 | tail -3')
sh(f'{R} mim install mmengine 2>&1 | tail -3')
sh(f'{R} mim install "mmcv==2.0.1" 2>&1 | tail -5')
sh(f'{R} mim install "mmdet==3.1.0" 2>&1 | tail -3')
sh(f'{R} mim install "mmpose==1.1.0" 2>&1 | tail -3')
LOG['install_s'] = round(time.perf_counter() - t, 1)
print(f"\ninstall took {LOG['install_s']}s")

# The gate run 1 lacked: prove every import resolves BEFORE downloading gigabytes.
print('\n--- imports, inside the env ---')
missing = []
for mod in ['torch', 'mmcv', 'mmpose', 'mmdet', 'diffusers', 'transformers', 'omegaconf']:
    r = sh(f'{R} python -c "import {mod}; print({mod}.__version__)"', quiet=True)
    ok = r.returncode == 0
    print(f'  {mod:14} {r.stdout.strip() if ok else "MISSING"}')
    if not ok:
        missing.append(mod)
LOG['missing_imports'] = missing
if missing:
    note(f'imports still missing after install: {missing} -- fix before continuing')
RUN = R


## 3B. Route B — stay on this Python

Run this **instead of 3A** if cell 2 said Route B, or if you want the 30-minute attempt
before committing to conda. Uses whichever `mmcv` has a wheel for this Python, which may
not match the `mmpose` API MuseTalk calls.


In [ ]:
WORK = Path('/content/m0')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)

if not (WORK / 'MuseTalk').exists():
    t = time.perf_counter()
    sh('git clone --depth 1 https://github.com/TMElyralab/MuseTalk.git', check=True)
    LOG['clone_s'] = round(time.perf_counter() - t, 1)
os.chdir(WORK / 'MuseTalk')
LOG['commit'] = sh('git rev-parse --short HEAD', quiet=True).stdout.strip()

t = time.perf_counter()
sh('pip install -q -r requirements.txt 2>&1 | tail -10')
sh('pip install -q -U openmim 2>&1 | tail -3')
sh('mim install mmengine 2>&1 | tail -3')
sh('mim install mmcv 2>&1 | tail -5')      # newest with a wheel for this Python
sh('mim install mmdet mmpose 2>&1 | tail -5')
LOG['install_s'] = round(time.perf_counter() - t, 1)

print('\n--- imports ---')
missing = []
for mod in ['torch', 'mmcv', 'mmpose', 'mmdet', 'diffusers', 'transformers', 'omegaconf']:
    r = sh(f'python -c "import {mod}; print({mod}.__version__)"', quiet=True)
    ok = r.returncode == 0
    print(f'  {mod:14} {r.stdout.strip() if ok else "MISSING"}')
    if not ok:
        missing.append(mod)
LOG['missing_imports'] = missing
if missing:
    note(f'Route B left {missing} missing -- this is the 3.12 wheel problem; take Route A')
RUN = ''   # no conda prefix on this route


## 4. Weights — downloaded, then **verified**

This is the cell whose absence cost run 1. MuseTalk's `download_weights.sh` points
`HF_ENDPOINT` at a mirror, has no `set -e`, and validates nothing, so a total failure still
exits 0. Here the endpoint is forced back to Hugging Face, and every expected checkpoint is
size-checked afterwards. **A missing file raises** rather than letting you measure nothing.


In [ ]:
# Expected files and a conservative floor for each, from the README's directory tree.
EXPECTED = {
    'models/musetalkV15/unet.pth': 100,
    'models/musetalkV15/musetalk.json': 0,
    'models/musetalk/pytorch_model.bin': 100,
    'models/musetalk/musetalk.json': 0,
    'models/sd-vae/diffusion_pytorch_model.bin': 100,
    'models/sd-vae/config.json': 0,
    'models/whisper/pytorch_model.bin': 50,
    'models/whisper/config.json': 0,
    'models/dwpose/dw-ll_ucoco_384.pth': 100,
    'models/face-parse-bisent/79999_iter.pth': 30,
    'models/face-parse-bisent/resnet18-5c106cde.pth': 30,
}


def audit():
    """Report every expected checkpoint. Catches LFS pointers and Drive HTML pages too."""
    rows, bad = [], []
    for rel, floor_mb in EXPECTED.items():
        p = Path(rel)
        if not p.is_file():
            rows.append((rel, 'ABSENT', floor_mb)); bad.append(rel); continue
        mb = p.stat().st_size / 1e6
        head = p.open('rb').read(40)
        if head.startswith(b'version https://git-lfs'):
            rows.append((rel, 'LFS POINTER', floor_mb)); bad.append(rel); continue
        if head.lstrip()[:15].lower().startswith(b'<!doctype html') or head.lstrip().startswith(b'<html'):
            rows.append((rel, 'HTML (Drive quota page)', floor_mb)); bad.append(rel); continue
        if mb < floor_mb:
            rows.append((rel, f'{mb:.1f}MB TOO SMALL', floor_mb)); bad.append(rel); continue
        rows.append((rel, f'{mb:.1f}MB ok', floor_mb))
    width = max(len(r[0]) for r in rows)
    for rel, status, floor_mb in rows:
        flag = '  ' if status.endswith('ok') else '!!'
        print(f'{flag} {rel:<{width}}  {status}' + (f'  (expect >{floor_mb}MB)' if flag == '!!' and floor_mb else ''))
    return bad


t = time.perf_counter()
# Force the real Hugging Face endpoint. The bundled script sets a mirror that is often
# unreachable from Colab, and its failures do not change the exit code.
os.environ.pop('HF_ENDPOINT', None)
sh(f'{RUN} pip install -q -U "huggingface_hub[cli]" gdown 2>&1 | tail -3')
script = next((s for s in ['download_weights.sh', 'scripts/download_weights.sh']
               if Path(s).is_file()), None)
if script is None:
    note('no download_weights.sh in this clone -- check the README for the current step')
else:
    sh(f'HF_ENDPOINT=https://huggingface.co bash {script} 2>&1 | tail -25')
LOG['weights_s'] = round(time.perf_counter() - t, 1)
LOG['weights_on_disk'] = sh('du -sh models 2>/dev/null', quiet=True).stdout.split()[0] if Path('models').exists() else '0'

print(f"\ndownload took {LOG['weights_s']}s, models/ is {LOG['weights_on_disk']}\n")
bad = audit()
LOG['missing_weights'] = bad

if bad:
    note(f'{len(bad)} checkpoint(s) missing or corrupt after download')
    raise SystemExit(
        f'\nSTOP. {len(bad)} checkpoint(s) bad -- see the !! rows above.\n'
        'Do NOT run inference: any timing it produced would measure a crash.\n'
        'Re-run this cell once (transient HF rate limits are common). If a file is still\n'
        'an LFS POINTER: apt-get install -y git-lfs && git lfs install.\n'
        'If it is HTML: the gdown Google Drive link hit its quota -- download\n'
        '79999_iter.pth by hand and place it at models/face-parse-bisent/.\n'
    )
print('\nall checkpoints present and plausibly sized.')


## 5. The correct v1.5 invocation

Straight from the README, with the four arguments run 1 was missing. Timed, and peak VRAM
sampled from `nvidia-smi` in a thread because inference runs as a subprocess -- run 1's
`peak_vram_mib: 3` is what proved nothing had touched the GPU.


In [ ]:
def poll_vram(stop, samples, interval=0.25):
    q = 'nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits'
    while not stop.is_set():
        r = subprocess.run(q, shell=True, capture_output=True, text=True)
        if r.returncode == 0 and r.stdout.strip():
            samples.append(int(r.stdout.strip().splitlines()[0]))
        time.sleep(interval)


def run_timed(cmd, label):
    samples, stop = [], threading.Event()
    w = threading.Thread(target=poll_vram, args=(stop, samples), daemon=True); w.start()
    print(f'--- {label} ---')
    t = time.perf_counter()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = time.perf_counter() - t
    stop.set(); w.join(timeout=2)
    print((r.stdout or '')[-2500:])
    if r.returncode != 0:
        print('STDERR:', (r.stderr or '')[-3000:], file=sys.stderr)
    rec = {'seconds': round(elapsed, 2),
           'peak_vram_mib': max(samples) if samples else None,
           'exit_code': r.returncode}
    print(f"{label}: {rec['seconds']}s, peak VRAM {rec['peak_vram_mib']} MiB, "
          f"exit {rec['exit_code']}")
    return rec


FFMPEG = str(Path(LOG['ffmpeg']).parent) if LOG.get('ffmpeg') else '/usr/bin'
INFER = (
    f"{RUN} python -m scripts.inference "
    f"--inference_config configs/inference/test.yaml "
    f"--result_dir ./results/v2 "
    f"--unet_model_path models/musetalkV15/unet.pth "
    f"--unet_config models/musetalkV15/musetalk.json "
    f"--version v15 "
    f"--ffmpeg_path {FFMPEG}"
).strip()
print('command:\n ', INFER, '\n')

LOG['inference_cold'] = run_timed(INFER, 'inference (cold)')
if LOG['inference_cold']['exit_code'] != 0:
    note('cold inference failed -- read the stderr above; do not record any fps number')
else:
    LOG['inference_warm'] = run_timed(INFER.replace('/v2', '/v2warm'), 'inference (warm)')


## 6. Measure the output

Render time divided by audio duration is the viability number. Below 1.0 is faster than
real time.


In [ ]:
def probe(path):
    q = ('ffprobe -v error -select_streams v:0 -show_entries '
         'stream=width,height,nb_frames,avg_frame_rate,duration -of json '
         f'"{path}"')
    r = subprocess.run(q, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        return {'error': r.stderr.strip()[-300:]}
    st = json.loads(r.stdout)['streams'][0]
    return {'resolution': f"{st.get('width')}x{st.get('height')}",
            'frames': int(st['nb_frames']) if st.get('nb_frames') else None,
            'duration_s': round(float(st['duration']), 2) if st.get('duration') else None}


outputs = sorted(Path('results').rglob('*.mp4')) if Path('results').exists() else []
print(f'{len(outputs)} rendered file(s)')
if not outputs:
    note('no output video -- nothing proven; do NOT record any fps number')
else:
    newest = max(outputs, key=lambda p: p.stat().st_mtime)
    LOG['output'] = {'path': str(newest), **probe(newest)}
    print(json.dumps(LOG['output'], indent=2))
    warm = LOG.get('inference_warm') or LOG.get('inference_cold')
    frames, dur = LOG['output'].get('frames'), LOG['output'].get('duration_s')
    if frames and warm and warm.get('seconds'):
        LOG['effective_fps'] = round(frames / warm['seconds'], 2)
        print(f"\neffective fps: {LOG['effective_fps']}  (includes per-run setup, so a floor)")
    if dur and warm and warm.get('seconds'):
        LOG['realtime_ratio'] = round(warm['seconds'] / dur, 2)
        verdict = 'FASTER than real time' if LOG['realtime_ratio'] < 1 else 'SLOWER than real time'
        print(f"render time / audio duration: {LOG['realtime_ratio']}  ({verdict})")
    from IPython.display import Video, display
    display(Video(str(newest), embed=True, width=480))


## 7. Identity preparation, timed separately

The architecturally significant split: face detection, parsing, and latent encoding of the
reference frames are one-time and offline. Slow here is fine, and it is the reason
first-frame latency can be low at conversation time. Maps straight onto `prepare_identity`
versus `push_audio` in `TalkingHeadRenderer`.


In [ ]:
rt = Path('scripts/realtime_inference.py')
if not rt.is_file():
    note('no scripts/realtime_inference.py -- identity prep not measured separately')
else:
    RT = (f"{RUN} python -m scripts.realtime_inference "
          f"--inference_config configs/inference/realtime.yaml "
          f"--result_dir ./results/rt "
          f"--unet_model_path models/musetalkV15/unet.pth "
          f"--unet_config models/musetalkV15/musetalk.json "
          f"--version v15 --fps 25 --ffmpeg_path {FFMPEG}").strip()
    LOG['realtime_first'] = run_timed(RT, 'realtime (prep + inference)')
    LOG['realtime_cached'] = run_timed(RT, 'realtime (prep cached)')
    a = LOG['realtime_first'].get('seconds'); b = LOG['realtime_cached'].get('seconds')
    if a and b and LOG['realtime_first']['exit_code'] == 0:
        LOG['identity_prep_s'] = round(a - b, 2)
        print(f"\nidentity preparation: ~{LOG['identity_prep_s']}s, one-time and offline")
    else:
        note('realtime runs did not both succeed -- identity_prep_s is not meaningful')


## 8. The block to paste back


In [ ]:
LOG['setup_notes'] = NOTES
LOG['finished_at'] = time.strftime('%Y-%m-%d %H:%M')

ran = (LOG.get('inference_cold', {}).get('exit_code') == 0
       and bool(LOG.get('output'))
       and (LOG.get('inference_cold', {}).get('peak_vram_mib') or 0) > 500)
LOG['inference_actually_ran'] = ran

print('=' * 70)
print(json.dumps(LOG, indent=2, default=str))
print('=' * 70)
if ran:
    print('\nInference ran: exit 0, an output file exists, and VRAM went past 500 MiB.')
    print('These numbers are real. Paste the block above back.')
else:
    print('\nInference did NOT run. Every timing above measures a failure, not a model.')
    print('Paste the block back anyway -- the setup_notes are the finding.')


---

## If it still fails

Fallbacks from `M0_SPIKE.md` §5, all still legitimate:

- **Ditto** (Apache-2.0, streaming-native). Its TensorRT requirement fights an ephemeral
  runtime, which is a documented cost rather than a surprise.
- **Smaller resolution**, and report the real number at that resolution.
- **Report the dead end.** "MuseTalk's documented stack does not install on the current
  free Colab runtime, and its weight downloader exits 0 having fetched 2% of the
  checkpoints" is a finding. §2.1's setup-fragility criterion is asking for exactly this.

The one thing that costs marks is a number nobody measured. `NOT YET MEASURED` is fine.
